In [1]:
import numpy as np
from collections import defaultdict


def cotangent(u, v):
    """计算两个向量之间的余切"""
    dot = np.dot(u, v)
    cross = np.linalg.norm(np.cross(u, v))
    return dot / cross if cross != 0 else 0


def compute_cotangent_weights_per_node(V, T):
    """
    计算每个节点的 one-ring 边的余切权重向量
    
    参数:
    V: ndarray of shape (N, 3), 每个节点的位置
    T: ndarray of shape (K, 3), 每个三角形单元的节点索引
    
    返回:
    C: list of lists, 每个节点的 one-ring 边的余切权重
    """
    N = V.shape[0]
    C = [[] for _ in range(N)]  # 每个节点的余切权重向量

    # 边到三角形的映射
    edge_to_triangles = defaultdict(list)

    for t_idx, tri in enumerate(T):
        edges = [
            (min(tri[0], tri[1]), max(tri[0], tri[1])),
            (min(tri[1], tri[2]), max(tri[1], tri[2])),
            (min(tri[2], tri[0]), max(tri[2], tri[0]))
        ]
        for edge in edges:
            edge_to_triangles[edge].append(t_idx)

    # 遍历每条边，计算余切权重
    edge_weights = {}
    for edge, triangles in edge_to_triangles.items():
        if len(triangles) == 2:  # 边需要属于两个三角形
            tri1, tri2 = triangles
            idx1, idx2 = T[tri1], T[tri2]
            
            # 找到不属于该边的顶点
            p0, p1 = edge
            p2_1 = list(set(idx1) - set(edge))[0]
            p2_2 = list(set(idx2) - set(edge))[0]

            # 顶点坐标
            v0, v1 = V[p0], V[p1]
            v2_1, v2_2 = V[p2_1], V[p2_2]

            # 向量计算
            cot_alpha = cotangent(v2_1 - v0, v2_1 - v1)
            cot_beta = cotangent(v2_2 - v0, v2_2 - v1)

            # 边的余切权重
            edge_weights[edge] = cot_alpha + cot_beta

    # 将边权重分配给节点
    for edge, weight in edge_weights.items():
        p0, p1 = edge
        C[p0].append(weight)
        C[p1].append(weight)

    return C

# 示例数据
V = np.array([
    [0, 0, 0],
    [1, 0, 0],
    [0, 0.8, 0],
    [1, 1, 0],
    [2, 1, 0],
    [1.9, 0, 0]
])  # 节点位置
T = np.array([
    [0, 1, 2],
    [1, 3, 2],
    [1, 3, 4],
    [1, 4, 5]
])  # 三角形单元

# 计算余切权重
# C = compute_cotangent_weights_per_node(V, T)

N = V.shape[0]
C = [[] for _ in range(N)]  # 每个节点的余切权重向量

# 边到三角形的映射
edge_to_triangles = defaultdict(list)

for t_idx, tri in enumerate(T):
    edges = [
        (min(tri[0], tri[1]), max(tri[0], tri[1])),
        (min(tri[1], tri[2]), max(tri[1], tri[2])),
        (min(tri[2], tri[0]), max(tri[2], tri[0]))
    ]
    for edge in edges:
        edge_to_triangles[edge].append(t_idx)

# 遍历每条边，计算余切权重
edge_weights = {}
for edge, triangles in edge_to_triangles.items():
    if len(triangles) == 2:  # 边需要属于两个三角形
        tri1, tri2 = triangles
        idx1, idx2 = T[tri1], T[tri2]
        
        # 找到不属于该边的顶点
        p0, p1 = edge
        p2_1 = list(set(idx1) - set(edge))[0]
        p2_2 = list(set(idx2) - set(edge))[0]

        # 顶点坐标
        v0, v1 = V[p0], V[p1]
        v2_1, v2_2 = V[p2_1], V[p2_2]

        # 向量计算
        cot_alpha = cotangent(v2_1 - v0, v2_1 - v1)
        cot_beta = cotangent(v2_2 - v0, v2_2 - v1)

        # 边的余切权重
        edge_weights[edge] = cot_alpha + cot_beta

# 将边权重分配给节点
for edge, weight in edge_weights.items():
    p0, p1 = edge
    C[p0].append(weight)
    C[p1].append(weight)

# 输出结果
for i, c in enumerate(C):
    print(f"节点 {i} 的余切权重向量: {c}")


节点 0 的余切权重向量: []
节点 1 的余切权重向量: [0.19999999999999996, 1.84, -0.10000000000000009]
节点 2 的余切权重向量: [0.19999999999999996]
节点 3 的余切权重向量: [1.84]
节点 4 的余切权重向量: [-0.10000000000000009]
节点 5 的余切权重向量: []


In [2]:
edge_weights

{(1, 2): 0.19999999999999996, (1, 3): 1.84, (1, 4): -0.10000000000000009}

In [7]:
# 将字典转换为双向图
graph = defaultdict(dict)
for (node1, node2), weight in edge_weights.items():
    graph[node1][node2] = weight
    graph[node2][node1] = weight  # 确保双向边

# 初始化用于存储结果的变量
node_neighbors = {}
node_weights = {}

# 填充每个节点的结果
for node, neighbors in graph.items():
    node_neighbors[node] = list(neighbors.keys())  # 一环邻居节点
    node_weights[node] = list(neighbors.values())  # 对应的权重

# 打印结果
print("Node Neighbors and Weights:")
for node in node_neighbors:
    print(f"Node {node}:")
    print(f"  Neighbors: {node_neighbors[node]}")
    print(f"  Weights: {node_weights[node]}")

Node Neighbors and Weights:
Node 1:
  Neighbors: [2, 3, 4]
  Weights: [0.19999999999999996, 1.84, -0.10000000000000009]
Node 2:
  Neighbors: [1]
  Weights: [0.19999999999999996]
Node 3:
  Neighbors: [1]
  Weights: [1.84]
Node 4:
  Neighbors: [1]
  Weights: [-0.10000000000000009]


In [6]:
graph

defaultdict(dict,
            {1: {2: 0.19999999999999996, 3: 1.84, 4: -0.10000000000000009},
             2: {1: 0.19999999999999996},
             3: {1: 1.84},
             4: {1: -0.10000000000000009}})

In [4]:
lhs = np.zeros((V.shape[0], V.shape[0]))
for q_i in range(V.shape[0]):
    if q_i not in node_neighbors or q_i not in node_weights:
        continue  # 跳过不存在于字典的节点

    # 获取邻居列表和对应权重
    neighbors = node_neighbors[q_i]
    weights = node_weights[q_i]

    weights_np = np.array(weights)
    weights_sum = np.sum(weights_np)

    weights_np_new = np.append(weights_np, -weights_sum)
    neighbors_new = neighbors + [q_i]

    lhs_tmp = np.outer(weights_np_new, weights_np_new)

    for m in neighbors_new:
        for n in neighbors_new:
            lhs[m, n] += lhs_tmp[neighbors_new.index(m), neighbors_new.index(n)]

lhs

array([[ 0.    ,  0.    ,  0.    ,  0.    ,  0.    ,  0.    ],
       [ 0.    ,  7.1992, -0.428 , -6.9552,  0.184 ,  0.    ],
       [ 0.    , -0.428 ,  0.08  ,  0.368 , -0.02  ,  0.    ],
       [ 0.    , -6.9552,  0.368 ,  6.7712, -0.184 ,  0.    ],
       [ 0.    ,  0.184 , -0.02  , -0.184 ,  0.02  ,  0.    ],
       [ 0.    ,  0.    ,  0.    ,  0.    ,  0.    ,  0.    ]])